# 02 — IDD Dataset Validation

Purpose:
- Validate Pascal VOC XML annotations.
- Check bounding-box coordinates.
- Identify invalid or empty annotations.
- Summarize dataset quality before YOLO label conversion.

In [1]:
# CELL 2: Imports and dataset paths
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd
import json

ROOT = Path("/workspace/IDD_Detection")
PROJECT = Path("/workspace/ITS_Project")

metadata = json.loads(
    (PROJECT / "configs" / "idd_metadata.json").read_text()
)

print("Dataset metadata loaded.")
print("Training images:", metadata["splits"]["train"])
print("Validation images:", metadata["splits"]["val"])
print("Test images:", metadata["splits"]["test"])

Dataset metadata loaded.
Training images: 31569
Validation images: 10225
Test images: 4794


In [2]:
# CELL 3: Prepare annotation lists
train_entries = [
    x.strip() for x in (ROOT / "train.txt").read_text().splitlines()
    if x.strip()
]
val_entries = [
    x.strip() for x in (ROOT / "val.txt").read_text().splitlines()
    if x.strip()
]

xml_files = {
    "train": [
        (ROOT / "Annotations" / x).with_suffix(".xml")
        for x in train_entries
    ],
    "val": [
        (ROOT / "Annotations" / x).with_suffix(".xml")
        for x in val_entries
    ],
}

print("Training XML paths:", len(xml_files["train"]))
print("Validation XML paths:", len(xml_files["val"]))

Training XML paths: 31569
Validation XML paths: 10225


In [3]:
# CELL 4: Validate XML structure and bounding-box coordinates
def validate_xml(xml_path):
    issues = []

    if not xml_path.is_file():
        return ["missing_xml"]

    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError:
        return ["malformed_xml"]

    objects = root.findall("object")
    if not objects:
        issues.append("no_objects")

    for obj in objects:
        box = obj.find("bndbox")

        if box is None:
            issues.append("missing_bndbox")
            continue

        try:
            xmin = float(box.findtext("xmin"))
            ymin = float(box.findtext("ymin"))
            xmax = float(box.findtext("xmax"))
            ymax = float(box.findtext("ymax"))
        except (TypeError, ValueError):
            issues.append("invalid_coordinates")
            continue

        if xmax <= xmin or ymax <= ymin:
            issues.append("nonpositive_box")

    return issues

print("Validation function ready.")

Validation function ready.


In [4]:
# CELL 5: Run validation on train and validation annotations
validation_results = []

for split, paths in xml_files.items():
    for path in paths:
        issues = validate_xml(path)
        validation_results.append({
            "split": split,
            "xml_path": str(path.relative_to(ROOT)),
            "issue_count": len(issues),
            "issues": ", ".join(issues)
        })

validation_df = pd.DataFrame(validation_results)

print("Annotations checked:", len(validation_df))
print("Annotations with issues:", (validation_df["issue_count"] > 0).sum())
display(validation_df[validation_df["issue_count"] > 0].head(20))

Annotations checked: 41794
Annotations with issues: 1


,split,xml_path,issue_count,issues
29410,train,Annotations/highquality_16k/HYD-2018-04-26_13-...,1,nonpositive_box
